In [2]:
from pathlib import Path
import pandas as pd

In [3]:
path_working_dir = Path("/home/b05b01002/HDD/project_scRNAed")
path_output = path_working_dir / "outputs/notebooks/Compare-minimap2-STAR"
path_mini_align_stats = path_working_dir / "outputs/notebooks/Compare-minimap2-STAR/mini_retag_10000.csv"
path_star_align_stats = path_working_dir / "outputs/notebooks/Compare-minimap2-STAR/star_umi_reads_10000.csv"


In [4]:
mini_align_stat = pd.read_csv(path_mini_align_stats, header=None, index_col=None)
mini_align_stat[0] = mini_align_stat[0].str.split(";").apply(lambda x: x[0])
mini_align_stat.index = mini_align_stat[0]
mini_align_stat.columns = ["qname", "flag", "chr", "pos", "mapq"]

In [5]:
star_align_stat = pd.read_csv(path_star_align_stats, header=None, index_col=None)
star_align_stat.index = star_align_stat[0]
star_align_stat.columns = ["qname", "flag", "chr", "pos", "mapq"]

In [6]:
star_align_stat.merge(
    mini_align_stat,
    left_index=True,
    right_index=True,
    how="left",
    suffixes=("_star", "_mini"),
).to_csv(
    path_output / "compare_mini_star_10000.csv",
    index=False,
    header=True,
    sep=",",
)

In [ ]:
counts = {
    "STAR yes; minimap2 no": 0,
    "STAR yes; minimap2 yes; suppl alignment (mapq > 30)": 0,
    "STAR yes; minimap2 yes; different chr": 0,
    "STAR yes; minimap2 yes; 0 < abs(delta pos) <= 20": 0,
    "STAR yes; minimap2 yes; abs(delta pos) > 20": 0,
    "STAR yes; minimap2 yes; different same pos": 0
}

for idx in star_align_stat.index:
    if idx not in mini_align_stat.index:
        counts["STAR yes; minimap2 no"] += 1
        continue
    else:
        mini_aligns = mini_align_stat.loc[idx,:]
        star_aligns = star_align_stat.loc[idx,:]
        
        # if supplementary alignments
        if mini_aligns.shape == (2, 5):
            supp_alignment = mini_aligns[(mini_aligns["flag"] == 2048) | (mini_aligns["flag"] == 2064)]
            prim_alignment = mini_aligns[(mini_aligns["flag"] == 0) | (mini_aligns["flag"] == 16)]
            if supp_alignment["mapq"].values[0] > 30:
                counts["STAR yes; minimap2 yes; suppl alignment (mapq > 30)"] += 1
                continue
            else:
                mini_aligns = prim_alignment.loc[idx,:]
        #             
        if mini_aligns["chr"] != star_aligns["chr"]:
            counts["STAR yes; minimap2 yes; different chr"] += 1
        elif mini_aligns["pos"] != star_aligns["pos"]:
            if abs(mini_aligns["pos"] - star_aligns["pos"]) <= 20:
                counts["STAR yes; minimap2 yes; 0 < abs(delta pos) <= 20"] += 1
            else:
                counts["STAR yes; minimap2 yes; abs(delta pos) > 20"] += 1
        elif mini_aligns["pos"] == star_aligns["pos"]:
            counts["STAR yes; minimap2 yes; different same pos"] += 1


In [10]:
sum([v for k,v in counts.items()])

10000

In [11]:
counts

{'STAR yes; minimap2 no': 3741,
 'STAR yes; minimap2 yes; suppl alignment (mapq > 30)': 14,
 'STAR yes; minimap2 yes; different chr': 229,
 'STAR yes; minimap2 yes; 0 < abs(delta pos) <= 20': 686,
 'STAR yes; minimap2 yes; abs(delta pos) > 20': 261,
 'STAR yes; minimap2 yes; different same pos': 5069}